<a href="https://colab.research.google.com/github/vifirsanova/llm-tutorial/blob/main/mas/response_formatting_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain-openai -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.7 MB/s eta 0:00:00


In [4]:
from google.colab import userdata

YANDEX_CLOUD_FOLDER = userdata.get('YANDEX_CLOUD_FOLDER')
YANDEX_CLOUD_API_KEY = userdata.get("YANDEX_API_KEY")
YANDEX_CLOUD_MODEL = "gpt-oss-120b/latest"

In [5]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Define schema
class MovieReview(BaseModel):
    title: str
    rating: int
    summary: str

# Use Yandex endpoint directly in Colab
llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",  # Replace with your folder ID
    api_key=YANDEX_CLOUD_API_KEY,  # Replace with your API key
    base_url="https://ai.api.cloud.yandex.net/v1",
)

structured_llm = llm.with_structured_output(MovieReview)
result = structured_llm.invoke("Review the movie Inception")
print(result.model_dump())

{'title': 'Inception – A Mind‑Bending Heist That Still Holds Up', 'rating': 4, 'summary': 'Christopher Nolan’s 2010 sci‑fi thriller is a masterclass in storytelling, visual design, and thematic depth. It rewards repeat viewings, although its dense plot can be a hurdle for casual viewers. For those who love puzzles, spectacular set‑pieces, and emotional stakes, it remains a modern classic.'}


In [8]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Literal

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(description="Issue category")
    urgency: Literal["low", "medium", "high", "critical"] = Field(description="Urgency level based on business impact")
    department: Literal["sales", "support", "engineering", "finance"] = Field(description="Responsible department")
    estimated_resolution_hours: int = Field(description="Estimated hours to resolve", ge=1, le=72)
    reason: str = Field(description="Brief reason for classification")

# Use Yandex endpoint
llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
)

# Structured output for ticket classification
structured_llm = llm.with_structured_output(TicketClassification)

# Example tickets with context
tickets = [
    "Customer was double-charged $299 for premium subscription. Transaction IDs: TXN-123, TXN-456. Customer called 3 times already.",
    "Server down for 2 hours, all EU customers affected. No ETA for fix yet.",
    "User wants to downgrade from enterprise to pro plan, but annual contract signed 2 months ago.",
]

for ticket in tickets:
    print(f"\nTicket: {ticket}")
    result = structured_llm.invoke(f"Classify this support ticket with context: {ticket}")
    print(f"Classification: {result.model_dump()}")
    print("-" * 50)


Ticket: Customer was double-charged $299 for premium subscription. Transaction IDs: TXN-123, TXN-456. Customer called 3 times already.
Classification: {'category': 'billing', 'urgency': 'high', 'department': 'finance', 'estimated_resolution_hours': 2, 'reason': 'duplicate_charge_premium_subscription_299_usd_multiple_contacts'}
--------------------------------------------------

Ticket: Server down for 2 hours, all EU customers affected. No ETA for fix yet.
Classification: {'category': 'technical', 'urgency': 'critical', 'department': 'engineering', 'estimated_resolution_hours': 2, 'reason': 'service outage – server down across EU region (all EU customers affected) – no ETA for fix yet'}
--------------------------------------------------

Ticket: User wants to downgrade from enterprise to pro plan, but annual contract signed 2 months ago.
Classification: {'category': 'billing', 'urgency': 'medium', 'department': 'sales', 'estimated_resolution_hours': 48, 'reason': 'downgrade_request_co

In [7]:
from langchain_openai import ChatOpenAI

# Use Yandex endpoint with JSON response format
llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    model_kwargs={"response_format": {"type": "json_object"}}
)

response = llm.invoke("List 3 colors in JSON format.")
print(response.content)

{
  "colors": [
    "red",
    "green",
    "blue"
  ]
}
